In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# Initialize data processors
from common.utils import DataPreprocessor, set_seed
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()



/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42
torch set to use deterministic algorithms


In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5, threshold=5,
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[[RARE], [RARE], [RARE], [RARE], [RARE]]",USA,[RARE],Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,[RARE],Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[[RARE], laura_linney, ernie_hudson_jr, tim_cu...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


In [17]:
({interaction_info_df.loc[0, "country"]} - {"[PAD]", "[RARE]"})

{'USA'}

In [ ]:
from tqdm import tqdm

# === 1. Dedup the table based on movieID ===
def deduplicate_movies(df):
    return df.drop_duplicates(subset='movieID').reset_index(drop=True)

# === 2. Exclude [PAD] and [RARE] in each attribute (actorID_idx, genre_idx, ...) ===
def clean_attributes(row, attr_cols, is_list):
    return {
        col: (set(row[col]) - {"[PAD]", "[RARE]"}) if is_list[i] else {row[col]}
        for i, col in enumerate(attr_cols)
    }

# === 3. Compute pairwise Jaccard similarity and average across attributes ===
def jaccard_similarity(set1, set2):
    if not set1 and not set2:
        return 0.0
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union != 0 else 0.0

def compute_similarity_matrix(df, attr_cols=['actorID', 'country', 'directorID', 'genre'], is_list=[True, False, False, True]):
    df = deduplicate_movies(df)
    movie_ids = sorted(df['movieID'].tolist())
    idx_mapping = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}
    n = len(idx_mapping)

    # Preprocess attributes into dict: movieID -> {attr_col: set}
    attr_data = {
        row['movieID']: clean_attributes(row, attr_cols, is_list)
        for _, row in df.iterrows()
    }

    # Initialize similarity matrix
    sim_matrix = np.zeros((n, n), dtype=np.float32)

    for i in tqdm(range(n), desc="Computing similarity matrix"):
        for j in range(i + 1, n):
            id_i, id_j = movie_ids[i], movie_ids[j]
            sims = []
            for attr in attr_cols:
                set_i = set(attr_data[id_i][attr])
                set_j = set(attr_data[id_j][attr])
                sims.append(jaccard_similarity(set_i, set_j))

            avg_sim = np.mean(sims)
            sim_matrix[i, j] = avg_sim
            sim_matrix[j, i] = avg_sim

    np.fill_diagonal(sim_matrix, 1.0)
    return sim_matrix, idx_mapping


In [19]:
sim_matrix, movie2idx = compute_similarity_matrix(interaction_info_df)


In [ ]:
# save files
np.savez_compressed(
    'artifacts/item_sim_data.npz',
    sim_matrix=sim_matrix,
    movie2idx=np.array(movie2idx, dtype=object)  # wrap the dict
)


In [33]:
# load files
data = np.load('artifacts/item_sim_data.npz', allow_pickle=True)
sim_matrix = data['sim_matrix']
movie2idx = data['movie2idx'].item()


In [34]:
print(sim_matrix.shape)
print(len(movie2idx))

(9461, 9461)
9461
